# 📍 Google Maps High-Throughput Scraper & Media Pipeline
### Production Workflow for Kaggle & Google Colab with Google Drive Storage

This notebook runs the automated Google Maps data extraction, strict geo-matching, and high-resolution photo pipeline.

**Key Capabilities:**
- 🔄 **100% Resumable & Fault-Tolerant:** Saves state to SQLite after every batch. If Kaggle/Colab disconnects or times out, re-running the cells continues exactly where it left off without duplicate queries or lost credits.
- ☁️ **Full Kaggle & Google Drive Integration:** Automatically detects source databases from Kaggle Datasets (`/kaggle/input/...`) or Google Drive (`/content/drive/MyDrive/...`), and persists outputs safely.
- 🎯 **Strict Multi-Factor Geo-Matching (SMPMS):** Evaluates top-5 candidates per venue with viewport coordinate locking (~200m radius), preventing false positive namesake collisions.
- 🏛️ **Enriched Venue Metadata:** Extracts neighborhoods (`neighborhood`), mall/complex locations (`located_in`), floors (`floor`), menus, 1-to-5 star rating distributions, Plus Codes, and Knowledge Graph IDs (`kgmid`).
- 🔍 **Optional Contacts & Social Media Enrichment:** Capable of extracting official business emails, phone numbers, and Instagram/Facebook profile metadata.
- 💬 **Rich Customer Feedback:** Extracts direct review URLs, reviewer profiles, and detailed context (dine-in/takeout, meal type, noise level, price per person).
- 🔑 **Multi-Key Auto-Rotation:** Cycles through multiple Apify API tokens, monitors USD credit balances in real time, and fails over smoothly when quotas are exhausted.
- 📸 **Original Quality WebP Media:** Downloads camera-original resolution photos (`=s0-rw`) concurrently.

## 1. Environment Detection & Google Drive Setup
Detects whether running on **Google Colab**, **Kaggle**, or **Local Environment** and sets up storage.

In [ ]:
import os
import sys
import glob

# Check runtime environment
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = 'kaggle_web_client' in sys.modules or os.path.exists('/kaggle/working')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive successfully mounted at /content/drive")
elif IS_KAGGLE:
    print("ℹ️ Running in Kaggle environment.")
    print("   • Working directory: /kaggle/working (High-speed NVMe)")
    print("   • Input datasets:    /kaggle/input")
else:
    print("ℹ️ Running in Local / Standard Python environment")

## 2. Install Dependencies
Install the required libraries (`apify-client`, `requests`, `tqdm`, `pandas`).

In [ ]:
# If running in Colab/Kaggle from scratch without local clone, uncomment below:
# !git clone https://github.com/osumy/apify_maps.git
# %cd apify_maps

!pip install -q apify-client requests tqdm pandas
print("✅ Dependencies installed successfully.")

## 3. Configuration & Paths Setup
Configure your Google Drive / Kaggle paths, operational parameters, and Apify API tokens.

> **Tip:** You can provide multiple Apify tokens. The key manager will automatically monitor balances and rotate between them.

In [ ]:
import os
import glob

# ---------------------------------------------------------------------------
# 1. Apify API Token Pool (Add your tokens here or via Colab/Kaggle Secrets)
# ---------------------------------------------------------------------------
colab_token = None
try:
    from google.colab import userdata
    colab_token = userdata.get('APIFY_TOKEN')
except Exception:
    pass

APIFY_TOKENS = colab_token or [
    "apify_api_***",
    "apify_api_***",
    "apify_api_***"
]

# ---------------------------------------------------------------------------
# 2. Output Storage Directory
# ---------------------------------------------------------------------------
if IS_COLAB:
    OUTPUT_DIR = "/content/drive/MyDrive/apify_maps_data"
elif IS_KAGGLE:
    # Working on high-speed /kaggle/working avoids SQLite network lock issues
    OUTPUT_DIR = "/kaggle/working/apify_maps_data"
else:
    OUTPUT_DIR = os.path.abspath("./data")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# 3. Smart Source Database Detection
# Automatically finds 'cafes_cafe_shard00.db' or uploaded Kaggle dataset
# ---------------------------------------------------------------------------
possible_source_paths = []
if IS_KAGGLE:
    possible_source_paths.extend(glob.glob("/kaggle/input/**/*cafes_cafe*.db", recursive=True))
    possible_source_paths.extend(glob.glob("/kaggle/input/**/*.db", recursive=True))
    possible_source_paths.extend(glob.glob("/kaggle/input/**/*.sqlite*", recursive=True))
possible_source_paths.extend([
    "/content/drive/MyDrive/apify_maps_data/cafes_cafe_shard00.db",
    os.path.join(OUTPUT_DIR, "cafes_cafe_shard00.db"),
    os.path.abspath("./data/cafes_cafe_shard00.db"),
    os.path.abspath("./cafes_cafe_shard00.db")
])

SOURCE_DB_PATH = None
for p in possible_source_paths:
    if os.path.exists(p) and not os.path.isdir(p):
        SOURCE_DB_PATH = p
        break

if not SOURCE_DB_PATH and IS_KAGGLE:
    kaggle_datasets = glob.glob("/kaggle/input/*/") + glob.glob("/kaggle/input/*/*/")
    for kd in kaggle_datasets:
        dbs = glob.glob(os.path.join(kd, "**/*.db"), recursive=True) + glob.glob(os.path.join(kd, "**/*.sqlite*"), recursive=True)
        if dbs:
            SOURCE_DB_PATH = dbs[0]
            break

if not SOURCE_DB_PATH:
    SOURCE_DB_PATH = os.path.join(OUTPUT_DIR, "cafes_cafe_shard00.db")

# Output database & file paths
# If SOURCE_DB_PATH points to a directory, resolve to the first DB file inside
if os.path.isdir(SOURCE_DB_PATH):
    dbs_in_dir = (glob.glob(os.path.join(SOURCE_DB_PATH, "**/*cafes_cafe*.db"), recursive=True) +
                  glob.glob(os.path.join(SOURCE_DB_PATH, "**/*.db"), recursive=True) +
                  glob.glob(os.path.join(SOURCE_DB_PATH, "**/*.sqlite*"), recursive=True))
    if dbs_in_dir:
        SOURCE_DB_PATH = dbs_in_dir[0]

OUTPUT_DB_PATH = os.path.join(OUTPUT_DIR, "google_maps.db")
RAW_JSONL_PATH = os.path.join(OUTPUT_DIR, "google_maps_raw.jsonl")
IMAGES_DIR = os.path.join(OUTPUT_DIR, "downloaded_images")

# ---------------------------------------------------------------------------
# 4. Operational & Enrichment Parameters
# ---------------------------------------------------------------------------
BATCH_SIZE = 50                 # Venues processed per Apify run
MAX_PLACES_PER_SEARCH = 5       # Candidates requested for strict multi-factor matching
MATCH_THRESHOLD = 0.80          # Minimum composite score for positive match
MAX_IMAGES_PER_PLACE = 50       # Photo URLs requested per venue
MAX_REVIEWS_PER_PLACE = 15      # Reviews requested per venue
SCRAPE_IMAGE_AUTHORS = True     # Capture image author & owner distinction
SCRAPE_CONTACTS = False         # Set True to extract emails & phones from websites
SCRAPE_SOCIAL_PROFILES = False  # Set True to enrich Instagram/Facebook profiles
MAX_IMAGE_WORKERS = 25          # Concurrent download threads for photos

print("Project Configuration Summary:")
print(f"  • Source DB:        {SOURCE_DB_PATH} ({'Found ✅' if os.path.exists(SOURCE_DB_PATH) else 'Pending Upload ⚠️'})")
print(f"  • Output DB:        {OUTPUT_DB_PATH}")
print(f"  • Raw JSONL:        {RAW_JSONL_PATH}")
print(f"  • Photos Dir:       {IMAGES_DIR}")
print(f"  • Batch Size:       {BATCH_SIZE}")
print(f"  • Enrich Contacts:  {SCRAPE_CONTACTS}")
print(f"  • Enrich Socials:   {SCRAPE_SOCIAL_PROFILES}")

## 4. Key Pool Balances & Health Check
Inspect all configured Apify tokens, their real-time USD credit balances, usage limits, and account health before running.

In [ ]:
from key_manager import ApifyKeyManager

km = ApifyKeyManager(tokens=APIFY_TOKENS, strategy="sequential")
km.print_summary()

## 5. Inspect Pending Venues (Resumability Check)
Checks how many total venues exist in the source database, how many have already been processed in `google_maps.db`, and how many remain pending.

In [ ]:
import sqlite3
import pandas as pd
from run_crawler import get_pending_cafes, init_output_db, resolve_data_path

SOURCE_DB_PATH = resolve_data_path(SOURCE_DB_PATH)

if not os.path.exists(SOURCE_DB_PATH):
    print(f"⚠️ Notice: Source database not found at '{SOURCE_DB_PATH}'.")
    print("If on Kaggle, add your source database via '+ Add Data' (Dataset).")
    print(f"If on Colab, upload 'cafes_cafe_shard00.db' to: {OUTPUT_DIR}/")
elif os.path.isdir(SOURCE_DB_PATH):
    print(f"⚠️ Notice: '{SOURCE_DB_PATH}' is a directory and no .db file was found inside.")
    print(f"Directory contents: {os.listdir(SOURCE_DB_PATH)}")
else:
    init_output_db(OUTPUT_DB_PATH)
    pending = get_pending_cafes(source_db_path=SOURCE_DB_PATH, output_db_path=OUTPUT_DB_PATH)
    print(f"📊 Total Pending Venues to Scrape: {len(pending)}")
    
    if pending:
        df_sample = pd.DataFrame(pending[:5])
        print("\nPreview of Next 5 Pending Venues:")
        display(df_sample[["cafe_id", "name", "latitude", "longitude", "city"]])

## 6. Run Google Maps Scraper
Executes the batched scraper with strict matching and comprehensive schema extraction.

- Set `MAX_RUN_TOTAL=10` or `50` for a test slice.
- Set `MAX_RUN_TOTAL=None` to scrape all remaining venues until completion.

In [ ]:
from run_crawler import run_scraper

# Set MAX_RUN_TOTAL = None for full production run, or an integer (e.g. 50) for a limited test
MAX_RUN_TOTAL = None  

crawl_stats = run_scraper(
    source_db_path=SOURCE_DB_PATH,
    output_db_path=OUTPUT_DB_PATH,
    raw_jsonl_path=RAW_JSONL_PATH,
    batch_size=BATCH_SIZE,
    max_total=MAX_RUN_TOTAL,
    max_images=MAX_IMAGES_PER_PLACE,
    max_reviews=MAX_REVIEWS_PER_PLACE,
    max_places_per_search=MAX_PLACES_PER_SEARCH,
    match_threshold=MATCH_THRESHOLD,
    scrape_image_authors=SCRAPE_IMAGE_AUTHORS,
    scrape_contacts=SCRAPE_CONTACTS,
    scrape_social_media_profiles=SCRAPE_SOCIAL_PROFILES,
    key_manager_instance=km,
    progress_bar=True
)

print("\nSession Finished! Summary:", crawl_stats)

## 7. Scraped Data Analytics & Verification
Query the SQLite database to verify data integrity, coverage of new fields (`neighborhood`, `located_in`, `reviews_distribution`), and customer review contexts.

In [ ]:
import sqlite3
import pandas as pd

if os.path.exists(OUTPUT_DB_PATH):
    conn = sqlite3.connect(OUTPUT_DB_PATH)
    
    print("--- Status Distribution ---")
    df_status = pd.read_sql_query("SELECT status, count(*) as count FROM places GROUP BY status", conn)
    display(df_status)
    
    print("\n--- Table Counts ---")
    for table in ["places", "place_images", "place_reviews", "place_attributes", "place_opening_hours", "place_popular_times"]:
        try:
            count = conn.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
            print(f"  • {table:22}: {count:,} rows")
        except Exception:
            pass
        
    print("\n--- New Enriched Fields Preview ---")
    df_enriched = pd.read_sql_query(
        "SELECT cafe_id, title, neighborhood, located_in, floor, total_score, "
        "reviews_distribution_5_star as five_stars, reviews_distribution_1_star as one_star, "
        "instagrams, website "
        "FROM places WHERE status = 'success' "
        "ORDER BY reviews_count DESC LIMIT 5",
        conn
    )
    display(df_enriched)
    
    print("\n--- Rich Review Contexts Sample ---")
    df_rev_context = pd.read_sql_query(
        "SELECT reviewer_name, stars, context_order_type, context_meal_type, "
        "context_price_per_person, context_noise_level, review_url "
        "FROM place_reviews WHERE review_context IS NOT NULL LIMIT 3",
        conn
    )
    display(df_rev_context)
    
    conn.close()
else:
    print(f"Database not found at '{OUTPUT_DB_PATH}' yet.")

## 8. Download High-Resolution Photos (`=s0-rw` WebP)
Downloads camera-master resolution WebP photos directly.

- **Resumable:** Database download statuses are committed periodically (every 100 images).
- `only_owner=True`: Downloads official cover photos and owner-uploaded images (cleanest quality).
- Set `only_owner=False` to download all photos including customer review uploads.

In [ ]:
from download_images import download_images

download_stats = download_images(
    output_db_path=OUTPUT_DB_PATH,
    images_dir=IMAGES_DIR,
    only_owner=True,           # True: owner photos only, False: all photos
    max_workers=MAX_IMAGE_WORKERS,
    batch_commit_size=100,
    progress_bar=True
)

print("\nDownload Session Summary:", download_stats)

## 9. Media & Storage Verification
Inspect disk space consumed by photos and preview sample downloaded WebP files.

In [ ]:
import glob
from IPython.display import Image, display

if os.path.exists(IMAGES_DIR):
    image_files = glob.glob(os.path.join(IMAGES_DIR, "*", "*.webp"))
    total_bytes = sum(os.path.getsize(f) for f in image_files)
    mb_size = total_bytes / (1024 * 1024)
    
    print(f"📸 Total Photos Downloaded: {len(image_files):,}")
    print(f"💾 Total Disk Space Used:    {mb_size:.2f} MB ({mb_size / 1024:.2f} GB)")
    
    if image_files:
        print("\nPreview of 3 Downloaded Photos:")
        for img_path in image_files[:3]:
            print(f"  Image: {img_path} ({os.path.getsize(img_path) / 1024:.1f} KB)")
            display(Image(filename=img_path, width=400))
else:
    print("No images directory found yet.")

## 10. Kaggle Export & Google Drive Synchronization
If you are running on **Kaggle**, use this section to archive and transfer your completed outputs to **Google Drive** or download them directly.

In [ ]:
import shutil

print("📦 Preparing Kaggle Output Artifacts...")

# 1. Create a compressed zip archive of output data for 1-click download from Kaggle UI
zip_output_path = "/kaggle/working/google_maps_crawled_data"
if os.path.exists(OUTPUT_DIR):
    print(f"Creating ZIP archive from: {OUTPUT_DIR} ...")
    shutil.make_archive(zip_output_path, 'zip', OUTPUT_DIR)
    zip_file = zip_output_path + ".zip"
    print(f"✅ Archive created: {zip_file} ({os.path.getsize(zip_file) / (1024*1024):.2f} MB)")
    print("You can download this archive directly from the Kaggle right-sidebar 'Output' section.")

# 2. Optional: Direct Google Drive Sync via PyDrive2 or Rclone
# If you have Google Drive API credentials or rclone configured on Kaggle, uncomment below:
# !pip install -q pydrive2
# from pydrive2.auth import GoogleAuth
# from pydrive2.drive import GoogleDrive
# ... upload zip_file to Google Drive folder